# Chapter 4 — Four-Port capstone

Engineer course · source candidate · CONVERGING

# Chapter 4 — Four-Port capstone

This complete Chapter is its own clean-kernel execution unit. Its six
web Lessons are reading views over these ordered source fragments; they
are not independent notebooks. QMD is the editable authority, while
`chapter.ipynb` is a generated zero-output transport artifact. The
required core ends after Lesson 3; Lessons 4–6 are optional
continuations on the same Plan.

## Lesson 1 — Build the original four-Port Plan

### Rebuild every physical subsystem

Follow the [engineer course setup](../setup.qmd), open this Chapter’s
`chapter.ipynb`, restart its Python kernel, and use **Run All**. Chapter
4 does not continue another Chapter’s state.

This is the exact physical declaration migrated from the four-Port
capstone: two 1 mm scalar CPW sections, a grounded readout, and a
floating pair. The finite-pi line sections are not exact distributed
equivalents.

In [ ]:
from IPython.display import display

from scnsim import (
    CircuitDiagramSpec,
    CircuitPlan,
    CircuitRun,
    DirectSolveSpec,
    ParameterDefinitions,
    ParameterSpec,
    RLGC,
    RLGCParameterSpec,
    ReductionPipeline,
    SParameterTrace,
    Theme,
    components,
    units as u,
)

plan = CircuitPlan(id="floating_probe_course")
feedline = plan.subsystem(id="feedline")
rlgc = RLGC(
    conductors=("signal",),
    reference_conductor="ground",
    resistance_per_length=[[0.0]] * u.ohm / u.m,
    inductance_per_length=[[420.0]] * u.nH / u.m,
    conductance_per_length=[[0.0]] * u.S / u.m,
    capacitance_per_length=[[175.0]] * u.pF / u.m,
)
inputs = ParameterDefinitions(id="floating_probe_design")
line_rlgc = inputs.parameter(
    id="line_rlgc",
    baseline=rlgc,
    spec=RLGCParameterSpec(
        conductors=("signal",),
        reference_conductor="ground",
    ),
)
readout_capacitance = inputs.parameter(
    id="readout_capacitance",
    baseline=110.0 * u.fF,
    spec=ParameterSpec(unit=u.fF),
)
readout_inductance = inputs.parameter(
    id="readout_inductance",
    baseline=5.8 * u.nH,
    spec=ParameterSpec(unit=u.nH),
)
mutual_capacitance = inputs.parameter(
    id="mutual_capacitance",
    baseline=16.0 * u.fF,
    spec=ParameterSpec(unit=u.fF),
)
mutual_inductance = inputs.parameter(
    id="mutual_inductance",
    baseline=7.0 * u.nH,
    spec=ParameterSpec(unit=u.nH),
)
plus_shunt_capacitance = inputs.parameter(
    id="plus_shunt_capacitance",
    baseline=45.0 * u.fF,
    spec=ParameterSpec(unit=u.fF),
)
minus_shunt_capacitance = inputs.parameter(
    id="minus_shunt_capacitance",
    baseline=42.0 * u.fF,
    spec=ParameterSpec(unit=u.fF),
)

left = feedline.add(
    components.transmission_line(
        id="left", length=1.0 * u.mm, rlgc=line_rlgc, n_sections=1
    )
)
right = feedline.add(
    components.transmission_line(
        id="right", length=1.0 * u.mm, rlgc=line_rlgc, n_sections=1
    )
)
input_bus = feedline.bus(id="input")
middle_bus = feedline.bus(id="middle")
output_bus = feedline.bus(id="output")
left_section = feedline.series(
    id="left_section",
    start=input_bus,
    elements=(
        left.between(
            left.pin("head", conductor="signal"),
            left.pin("tail", conductor="signal"),
        ),
    ),
    end=middle_bus,
)
right_section = feedline.series(
    id="right_section",
    start=middle_bus,
    elements=(
        right.between(
            right.pin("head", conductor="signal"),
            right.pin("tail", conductor="signal"),
        ),
    ),
    end=output_bus,
)
feedline_input_pin = feedline.expose_pin(id="input", at=input_bus)
feedline_coupling_pin = feedline.expose_pin(id="tap", at=middle_bus)
feedline_output_pin = feedline.expose_pin(id="output", at=output_bus)

The CPW reference conductor stays RLGC metadata. The three exposed
signal pins are the only parent-facing feedline boundaries.

In [ ]:
readout = plan.subsystem(id="readout")
readout_capacitor = readout.add(
    components.capacitor(
        id="capacitor",
        capacitance=readout_capacitance,
    )
)
readout_inductor = readout.add(
    components.inductor(
        id="inductor",
        inductance=readout_inductance,
    )
)
readout_bus = readout.bus(id="node")
readout.parallel(
    id="parallel_lc",
    start=readout_bus,
    branches=((readout_capacitor,), (readout_inductor,)),
    end=readout.ground,
)
readout_terminal = readout.expose_pin(id="readout_node", at=readout_bus)

floating = plan.subsystem(id="floating")
plus_bus = floating.bus(id="plus")
minus_bus = floating.bus(id="minus")
mutual_cap = floating.add(
    components.capacitor(id="mutual_cap", capacitance=mutual_capacitance)
)
mutual_ind = floating.add(
    components.inductor(id="mutual_ind", inductance=mutual_inductance)
)
plus_shunt = floating.add(
    components.capacitor(
        id="plus_shunt",
        capacitance=plus_shunt_capacitance,
    )
)
minus_shunt = floating.add(
    components.capacitor(
        id="minus_shunt",
        capacitance=minus_shunt_capacitance,
    )
)
floating.parallel(
    id="mutual_network",
    start=plus_bus,
    branches=((mutual_cap,), (mutual_ind,)),
    end=minus_bus,
)
plus_branch = floating.branch(
    id="plus_shunt",
    at=plus_bus,
    elements=(plus_shunt,),
    end=floating.ground,
)
minus_branch = floating.branch(
    id="minus_shunt",
    at=minus_bus,
    elements=(minus_shunt,),
    end=floating.ground,
)
plus_1 = floating.expose_pin(id="plus_1", at=plus_bus)
plus_2 = floating.expose_pin(id="plus_2", at=plus_bus)
minus_1 = floating.expose_pin(id="minus_1", at=minus_bus)
minus_2 = floating.expose_pin(id="minus_2", at=minus_bus)

The floating subsystem exposes two Pins for each of its two intrinsic
nodes. Those aliases do not create extra coordinates or merge distinct
parent-local wiring groups.

In [ ]:
feedline_in_bus = plan.bus(id="feedline_in")
feedline_out_bus = plan.bus(id="feedline_out")
readout_root_bus = plan.bus(id="readout_node")
floating_plus_bus = plan.bus(id="floating_plus")
floating_minus_bus = plan.bus(id="floating_minus")
floating_plus = floating_plus_bus.node
floating_minus = floating_minus_bus.node

plan.link(
    id="feedline_input_child",
    endpoints=(feedline_in_bus, feedline_input_pin),
)
plan.link(
    id="feedline_output_child",
    endpoints=(feedline_out_bus, feedline_output_pin),
)
plan.link(id="readout_child", endpoints=(readout_root_bus, readout_terminal))
plan.link(
    id="floating_plus_child",
    endpoints=(floating_plus_bus, plus_2),
)
plan.link(
    id="floating_minus_child",
    endpoints=(floating_minus_bus, minus_2),
)

feedline_coupler = plan.add(
    components.capacitor(
        id="feedline_readout_coupler",
        capacitance=6.0 * u.fF,
    )
)
plus_coupler = plan.add(
    components.capacitor(
        id="readout_to_floating_plus",
        capacitance=4.0 * u.fF,
    )
)
minus_coupler = plan.add(
    components.capacitor(
        id="readout_to_floating_minus",
        capacitance=3.0 * u.fF,
    )
)
feedline_readout = plan.series(
    id="feedline_readout",
    start=feedline_coupling_pin,
    elements=(feedline_coupler,),
    end=readout_root_bus,
)
readout_floating_plus = plan.series(
    id="readout_floating_plus",
    start=readout_root_bus,
    elements=(plus_coupler,),
    end=plus_1,
)
readout_floating_minus = plan.series(
    id="readout_floating_minus",
    start=readout_root_bus,
    elements=(minus_coupler,),
    end=minus_1,
)

feedline_in_port = plan.add_port(
    id="feedline_in",
    at=feedline_in_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
feedline_out_port = plan.add_port(
    id="feedline_out",
    at=feedline_out_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
probe_plus = plan.add_port(
    id="floating_probe_plus",
    at=floating_plus_bus,
    role="nonloading_probe",
    reference_impedance=50.0 * u.ohm,
)
probe_minus = plan.add_port(
    id="floating_probe_minus",
    at=floating_minus_bus,
    role="nonloading_probe",
    reference_impedance=50.0 * u.ohm,
)

The original Plan has four declared Ports: two terminated feedline
boundaries and two nonloading probe loads. PTC later chooses an
analytical View; it does not delete these Ports or change this physical
declaration.

## Lesson 2 — Review the explicit physical composition

### Compose the existing Plan without changing it

The complete Default layout remains outside this lesson’s success claim.
This is the established explicit probe-tee composition for the exact
Plan above. Its Ts, elbow, axes, sides, and local grounds are
presentation choices, not electrical nodes or an alternate model.

In [ ]:
from scnsim import DiagramAxis, DiagramSide, SchematicComposition

composition = SchematicComposition(plan=plan)
composition.axis(feedline, DiagramAxis.VERTICAL)
composition.axis(left_section, DiagramAxis.VERTICAL)
composition.axis(right_section, DiagramAxis.VERTICAL)
composition.order(plan, members=(feedline, readout, floating))

wiring = composition.replace_wiring(at=readout_terminal)
readout_t = wiring.tee(id="readout_attachment", branch=DiagramSide.BOTTOM)
split_t = wiring.tee(id="floating_split", branch=DiagramSide.BOTTOM)
lower_turn = wiring.elbow(
    id="floating_minus_turn",
    sides=(DiagramSide.TOP, DiagramSide.RIGHT),
)
wiring.connect(
    composition.endpoint(feedline_readout, boundary="end"),
    readout_t.left,
)
wiring.connect(readout_t.bottom, composition.endpoint(readout_terminal))
wiring.connect(readout_t.right, split_t.left)
wiring.connect(
    split_t.right,
    composition.endpoint(readout_floating_plus, boundary="start"),
)
wiring.connect(split_t.bottom, lower_turn.top)
wiring.connect(
    lower_turn.right,
    composition.endpoint(readout_floating_minus, boundary="start"),
)

composition.terminal_side(plus_1, DiagramSide.LEFT)
composition.terminal_side(minus_1, DiagramSide.LEFT)
composition.terminal_side(plus_2, DiagramSide.RIGHT)
composition.terminal_side(minus_2, DiagramSide.RIGHT)
composition.port_orientation(
    feedline_in_port,
    boundary_side=DiagramSide.TOP,
    load_side=DiagramSide.LEFT,
)
composition.port_orientation(
    feedline_out_port,
    boundary_side=DiagramSide.BOTTOM,
    load_side=DiagramSide.LEFT,
)
composition.port_orientation(
    probe_plus,
    boundary_side=DiagramSide.RIGHT,
    load_side=DiagramSide.TOP,
)
composition.port_orientation(
    probe_minus,
    boundary_side=DiagramSide.RIGHT,
    load_side=DiagramSide.BOTTOM,
)
composition.ground_side(plus_branch, DiagramSide.TOP)
composition.ground_side(minus_branch, DiagramSide.BOTTOM)

plus_wiring = composition.replace_wiring(at=floating_plus_bus)
plus_output = plus_wiring.straight(
    id="probe_attachment",
    axis=DiagramAxis.HORIZONTAL,
)
plus_wiring.connect(composition.endpoint(plus_2), plus_output.left)
plus_wiring.connect(plus_output.right, composition.endpoint(probe_plus))

minus_wiring = composition.replace_wiring(at=floating_minus_bus)
minus_output = minus_wiring.straight(
    id="probe_attachment",
    axis=DiagramAxis.HORIZONTAL,
)
minus_wiring.connect(composition.endpoint(minus_2), minus_output.left)
minus_wiring.connect(minus_output.right, composition.endpoint(probe_minus))

Every replaced group includes all its actual attachments. The equivalent
floating Pins share voltage through the child’s real internal wiring;
the composition does not shortcut their distinct parent-local groups.

In [ ]:
capstone_diagram = plan.render_schematic(
    CircuitDiagramSpec(
        representation="authoring",
        theme=Theme.AUTO,
        show_parameter_values=True,
        show_provenance=True,
        layout=composition,
    )
)
capstone_diagram.show()
capstone_diagram.audit.show()

The certificate binds the Plan and presentation. It does not claim that
PTC or any numerical request changed the drawing.

## Lesson 3 — Select the probe-compensated two-Port View

### Compare the original loaded View with its PTC derivation

The raw View includes all four declared loads. PTC removes only the two
named probe load stamps, and `retain()` selects the two feedline
coordinates for the requested scattering boundary.

In [ ]:
run = CircuitRun(plan=plan, workspace="workspaces/engineer-chapter-04")
raw_loaded_view = run.original
raw_transmission_trace = SParameterTrace(
    id="transmission",
    input_port="feedline_in",
    input_mode=(),
    output_port="feedline_out",
    output_mode=(),
)
raw_direct_spec = DirectSolveSpec(
    frequencies=[5.5, 6.0, 6.5] * u.GHz,
    traces=(raw_transmission_trace,),
)

ptc_view = raw_loaded_view.reduce(
    ReductionPipeline()
    .ptc(probe_plus, probe_minus)
    .retain("feedline_in", "feedline_out")
)

ptc_transmission_trace = SParameterTrace(
    id="transmission",
    input_port="feedline_in",
    input_mode=(),
    output_port="feedline_out",
    output_mode=(),
)
ptc_direct_spec = DirectSolveSpec(
    frequencies=[5.5, 6.0, 6.5] * u.GHz,
    traces=(ptc_transmission_trace,),
)
run.explain(raw_loaded_view, raw_direct_spec).show()
run.explain(ptc_view, ptc_direct_spec).show()

These three frequencies are exact declared samples, not a continuous
scan or a resonance locator.

In [ ]:
raw_direct = run.solve(raw_loaded_view, raw_direct_spec)
raw_transmission = raw_direct.traces["transmission"]
ptc_direct = run.solve(ptc_view, ptc_direct_spec)
ptc_transmission = ptc_direct.traces["transmission"]
display(raw_direct.s.view)
raw_transmission.show(magnitude="db", theme=Theme.AUTO)
display(ptc_direct.s.view)
ptc_transmission.show(magnitude="db", theme=Theme.AUTO)

> **Core stopping point**
>
> Stop here for the Chapter 4 core: one original four-Port physical
> Plan, its certified explicit diagram, the raw four-Port View’s
> feedline Direct Result, and the separately labeled PTC-selected
> two-Port Direct Result. The remaining Lessons are optional analytical
> continuations on this same Plan.

## Lesson 4 — Optional: transform the floating coordinates

### Branch another View from the same original Plan

This optional path repeats PTC, replaces the ordered public
floating-node pair with `floating.common` and `floating.differential`,
then retains the two feedline coordinates plus the differential
coordinate. It neither mutates the core View nor creates another
physical Plan.

In [ ]:
transformed_view = raw_loaded_view.reduce(
    ReductionPipeline()
    .ptc(probe_plus, probe_minus)
    .transform_pair(
        floating_plus,
        floating_minus,
        id="floating",
    )
    .retain(
        "feedline_in",
        "feedline_out",
        "floating.differential",
    )
)
transform_explanation = run.explain(
    transformed_view,
    DirectSolveSpec(frequencies=[5.5, 6.0, 6.5] * u.GHz),
)
display(transform_explanation.evidence)
transform_explanation.show()

The transform is an analysis basis change with recorded lineage. It does
not rename the two physical floating buses or alter the schematic.

## Lesson 5 — Optional: evaluate one Direct response scalar

### Read S21 at one explicitly declared frequency

`ResponseElementSpec` asks for one complex S element on the transformed
retained View. The declared 6.2 GHz point is evaluated directly; it is
not interpolated from either three-sample trace and is not a root, dip,
or qubit frequency.

In [ ]:
from scnsim import ResponseElementSpec

transmission_at_6_2_spec = ResponseElementSpec(
    family="S",
    input_coordinate="feedline_in",
    output_coordinate="feedline_out",
    frequency=6.2 * u.GHz,
)
run.explain(transformed_view, transmission_at_6_2_spec).show()
transmission_at_6_2 = run.evaluate(
    transformed_view,
    transmission_at_6_2_spec,
)
display(transmission_at_6_2.value)
display(transmission_at_6_2.magnitude)
display(transmission_at_6_2.real)
display(transmission_at_6_2.imag)
transmission_at_6_2.show()

## Lesson 6 — Optional: read independent Direct and pump-off HB Results

### Declare separate requests on the same transformed View

Direct and pump-off HB use the same selected View and named physical
channel, but they retain their own frequency grids and mode conventions.
This lesson does not interpolate, align, overlay, or compute a residual
between them.

In [ ]:
from scnsim import (
    CurrentDrive,
    HBCaseSpec,
    HBSolveSpec,
    HBTruncation,
    PumpAxis,
)

optional_direct_trace = SParameterTrace(
    id="transmission",
    input_port="feedline_in",
    input_mode=(),
    output_port="feedline_out",
    output_mode=(),
)
optional_direct_spec = DirectSolveSpec(
    frequencies=[5.5, 6.0, 6.5] * u.GHz,
    traces=(optional_direct_trace,),
)

pump = PumpAxis(id="pump", frequency=9.0 * u.GHz)
pump_drive = CurrentDrive(
    id="pump_drive",
    at=feedline_in_port,
    mode=(1,),
)
hb_trace = SParameterTrace(
    id="transmission",
    input_port="feedline_in",
    input_mode=(0,),
    output_port="feedline_out",
    output_mode=(0,),
)
hb_spec = HBSolveSpec(
    pump_axes=(pump,),
    drives=(pump_drive,),
    frequencies=[5.4, 5.9, 6.4] * u.GHz,
    cases=(HBCaseSpec(id="pump_off", currents={}),),
    truncation=HBTruncation(
        pump_harmonics=(3,),
        modulation_harmonics=(1,),
        three_wave_mixing=False,
        four_wave_mixing=True,
    ),
    traces=(hb_trace,),
)
run.explain(transformed_view, optional_direct_spec).show()
run.explain(transformed_view, hb_spec).show()

The exact-zero `pump_off` case is still an HB request with its own
lattice, truncation, result identity, and typed outcome.

In [ ]:
optional_direct = run.solve(transformed_view, optional_direct_spec)
hb = run.solve(transformed_view, hb_spec)

In [ ]:
optional_direct.traces["transmission"].show(
    magnitude="db",
    theme=Theme.AUTO,
)
display(optional_direct.s.view)

pump_off = hb.cases["pump_off"]
if pump_off.succeeded:
    pump_off.traces["transmission"].show(
        magnitude="db",
        theme=Theme.AUTO,
    )
    display(pump_off.s.view)
    display(pump_off.states)
    display(pump_off.state_node_map)
else:
    display(pump_off.failure)

The two Results answer their own declared requests. Similar-looking
values do not establish cross-backend closure, and a valid failed HB
case is not a trace.